<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_1_Classification_Basics/18_1_6_ccfraud_with_nested_crosseval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Nested Cross-Validation with Grid-Search

Since we haven't done cross-validation so far, we should be concerned that our results won't apply to other train-test splits. To really be confident, we should run nested-cross validation when we conduct our grid search. The downside to this is it will take a very long time. However, if this were a real world scenario and we were working for a large credit card company our additional time would be well spent to be confident in our findings.


### Why this is different
* The Outer Loop (cross_val_score): It splits your data into 5 parts. For each part, it hands the other 4 parts to the Grid Search to train on.
* The Inner Loop (GridSearchCV): Inside those 4 parts, it does its own 3-fold split to find the best parameters.
* The Result: We get 5 separate scores. If these scores are wildly different, it means our tuning is unstable or our dataset is too small. If they are consistent, we can be very confident in that $18,300 estimate you found earlier.
* Warning on Time: Since you have 18 combinations in your grid, a 3-fold inner CV, and a 5-fold outer CV, you are now training 270 models.  




In [40]:
# this takes 32 minutes to run, so I'm putting it into a conditional
# if you want to run it flip the False to True

# if False:
if True:

  # 1. Define the OBJECTIVE (Unchanged)
  def cost_weighted_objective(y_true, y_pred):
      # Use the globally defined cost constants
      cost_fp = COST_FP
      cost_fn = COST_FN
      p = 1.0 / (1.0 + np.exp(-y_pred))
      grad = cost_fn * (p - 1.0) * y_true + cost_fp * p * (1.0 - y_true)
      hess = (cost_fn * y_true + cost_fp * (1.0 - y_true)) * (p * (1.0 - p))
      return grad, hess

  # 2. Define the SCORER (Unchanged)
  def total_cost_scorer(y_true, y_prob):
      y_pred = (y_prob >= 0.95).astype(int)
      fp = np.sum((y_pred == 1) & (y_true == 0))
      fn = np.sum((y_pred == 0) & (y_true == 1))
      return -(fp * COST_FP + fn * COST_FN)

  # 3. Setup Nested CV Structure
  # Inner CV: Used to find the best hyperparameters
  inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

  # Outer CV: Used to estimate the true performance (generalization error)
  outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

  # Define the Inner Search
  grid_search = GridSearchCV(
      estimator=xgb.XGBClassifier(
          objective=cost_weighted_objective,
          tree_method='hist',
          enable_categorical=True,
          random_state=42
      ),
      param_grid={
          'max_depth': [3, 5, 7],
          'n_estimators': [100, 200, 300],
          'learning_rate': [0.01, 0.1]
      },
      scoring='average_precision',
      cv=inner_cv
  )

  # 4. Run Nested CV
  # This executes the inner search multiple times (once for each outer fold)
  nested_scores = cross_val_score(grid_search, X, y, cv=outer_cv, scoring='average_precision')

  print(f"Nested CV PR-AUC Scores: {nested_scores}")
  print(f"Mean PR-AUC Score: {nested_scores.mean():.4f}")
  print(f"Standard Deviation: {nested_scores.std():.4f}")


Nested CV PR-AUC Scores: [0.84036927 0.88905046 0.86399462 0.86072712 0.82716138]
Mean PR-AUC Score: 0.8563
Standard Deviation: 0.0212


### Nested CV Results for Discussion


Nested CV is designed to tell us how well our tuning process works, not to give us a single final model. Because the inner loop runs a new grid search for each outer fold, it actually finds a (possibly different) "best" set of parameters each time.

The results above are PR-AUC scores on data the model has never seen, averaged across 5 outer folds.

What these numbers mean in practical terms:

1. **Consistent "Real World" Performance.** The five numbers in the brackets are the PR-AUC values for each of the 5 outer test folds. If they were wildly different, we would worry that our tuning process is unstable. If they are close together, we can trust that the mean is a reliable estimate of what to expect on future data.
2. **Reliability (Low Standard Deviation).** A low standard deviation relative to the mean tells us the model is stable. It isn't just getting "lucky" on specific parts of the data.
3. **Our Tuning Worked (Hopefully).** If the Nested CV mean PR-AUC is higher than the PR-AUC we observed on the single held-out test split earlier, the hyperparameter tuning (GridSearch) genuinely found an improvement that generalizes.
4. **PR-AUC is Threshold-Independent.** Notice we are no longer talking about dollars here. PR-AUC measures how well the model separates the two classes across all possible thresholds. Once we are confident in the model, we pick a threshold separately using the cost-based approach from earlier.

**Key takeaway:** Nested CV gives us an honest estimate of how our tuning pipeline performs on unseen data. Once we are satisfied with the estimate, we can re-run the grid search to get a final set of parameters for production.

### How to get the final "Ideal" parameters
To get the actual parameters you will use in production, you must run the GridSearch one last time on your entire dataset (or your full training set).


In [41]:
if not grid_search:
  print('Running this cell requires you to have run the previous cell.')

else:
  # 1. Run the search on the FULL dataset to get the final production model
  grid_search.fit(X, y)

  # 2. Now we can see the "Ideal" parameters
  print("Ideal Parameters:", grid_search.best_params_)

  # 3. And get the final model
  final_model = grid_search.best_estimator_

  # 4. Save the model - This is what would get used in production
  final_model.get_booster().save_model("fraud_model_v1.json")

Ideal Parameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 300}


We didn't do much of a grid search here. We could further explore by looking at lower learning rates and higher numbers of estimators.

## Evaluating the Tuned Model on the Test Set

Now let's compare the tuned model's performance against our original baseline model on the held-out test set.

In [42]:
# Evaluate the tuned model (from GridSearch) on the test set
best_model = grid.best_estimator_
y_proba_tuned = best_model.predict_proba(X_test)[:, 1]
y_pred_tuned = (y_proba_tuned >= cost_optimal_threshold).astype(int)

print(f"--- Tuned Model (threshold={cost_optimal_threshold:.2f}) ---")
print(classification_report(y_test, y_pred_tuned, target_names=['Not-Fraud (0)', 'Fraud (1)']))

# Compare costs: baseline vs tuned
fp_baseline = ((y_pred == 1) & (y_test == 0)).sum()
fn_baseline = ((y_pred == 0) & (y_test == 1)).sum()
cost_baseline = fp_baseline * COST_FP + fn_baseline * COST_FN

fp_tuned = ((y_pred_tuned == 1) & (y_test == 0)).sum()
fn_tuned = ((y_pred_tuned == 0) & (y_test == 1)).sum()
cost_tuned = fp_tuned * COST_FP + fn_tuned * COST_FN

print(f"\n--- Cost Comparison (FP=$100, FN=$450) ---")
print(f"Baseline model (default 0.5 threshold): ${cost_baseline:,.0f}")
print(f"Tuned model (threshold={cost_optimal_threshold:.2f}):     ${cost_tuned:,.0f}")
print(f"Savings: ${cost_baseline - cost_tuned:,.0f}")

# 5. Final Comparison Table
comparison_data = {
    'Model': ['Baseline (Untuned)', 'Tuned Model'],
    'Threshold': [0.50, f"{cost_optimal_threshold:.4f}"],
    'Total Cost ($)': [cost_baseline, cost_tuned],
    'Savings ($)': [0, cost_baseline - cost_tuned]
}
comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

--- Tuned Model (threshold=0.10) ---
               precision    recall  f1-score   support

Not-Fraud (0)       1.00      1.00      1.00     85295
    Fraud (1)       0.71      0.82      0.76       148

     accuracy                           1.00     85443
    macro avg       0.86      0.91      0.88     85443
 weighted avg       1.00      1.00      1.00     85443


--- Cost Comparison (FP=$100, FN=$450) ---
Baseline model (default 0.5 threshold): $17,600
Tuned model (threshold=0.10):     $17,050
Savings: $550


## Summary and Key Takeaways

In this notebook, we walked through a complete fraud detection pipeline:

1. **Data Loading & Exploration:** Loaded the credit card fraud dataset (284,807 transactions, only 0.17% fraud) and verified the extreme class imbalance.

2. **Baseline XGBoost Model:** Trained an initial model with default class weighting. Even untuned, the model showed strong discriminative ability.

3. **Evaluation Metrics:**
   - **Confusion Matrix** helped us see the tradeoff between false positives (annoying customers) and false negatives (missing fraud).
   - We deliberately **skipped ROC AUC** because it can be misleadingly optimistic on datasets with this much class imbalance.
   - **Precision-Recall AUC** gave us a more honest picture of model performance on the minority class.

4. **Threshold Tuning:** We compared several threshold selection strategies:
   - Default (0.5)
   - Youden's J (statistical)
   - F-Beta optimal (balancing precision and recall)
   - **Cost-optimal** (minimizing total business cost) -- the most practically useful approach.

5. **Hyperparameter Tuning:** Used GridSearchCV scored by PR-AUC to find parameters that do the best job of separating fraud from non-fraud.

6. **Nested Cross-Validation:** Verified that our tuning process generalizes reliably across different data splits.

**Key insight:** In real-world classification problems, the "best" model isn't always the one with the highest accuracy or F1 score. It's the one that best aligns with your business objectives. Cost-based threshold selection lets you directly translate model decisions into dollar impacts.